# 4.15 Independent Component Analysis

Independent Component Analysis rotates whitened data toward statistically independent, non-Gaussian sources.

Part 4 moves from prediction with labels to discovery without labels. Vectors, covariance, kernels, and matrix factorization become the language for hidden low-dimensional structure. Geometry supplies distances, neighborhoods, projections, and matrix shapes; optimization decides which structure is kept.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.datasets import load_digits
from sklearn.decomposition import FactorAnalysis
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import Lasso
from sklearn.manifold import trustworthiness
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(7)


def dimred_ladder():
    """D1..D5 dimensionality-reduction ladder. Returns [(name, X, y), ...] of rising ambient dim.

    2-D toy -> 3-D swiss-roll-ish -> digits(64-D) -> the same with noise dims -> a wide feature set.
    y is a color/label for visualization only.
    """
    rungs = []
    rng = np.random.default_rng(3)

    t = np.linspace(0, 4, 120)
    x1 = np.column_stack([t, 0.5 * t + rng.normal(0, 0.05, 120)])
    rungs.append(("D1 near-1-D line in 2-D", x1, t))

    tt = np.linspace(0, 3 * np.pi, 200)
    x2 = np.column_stack([tt * np.cos(tt), 8 * rng.random(200), tt * np.sin(tt)])
    rungs.append(("D2 swiss-roll (3-D)", x2, tt))

    digits = load_digits()
    rungs.append(("D3 digits (real, 64-D)", digits.data / 16.0, digits.target))

    xn = np.hstack([digits.data / 16.0, rng.normal(0, 1, size=(digits.data.shape[0], 32))])
    rungs.append(("D4 digits + 32 noise dims", xn, digits.target))

    bc = load_breast_cancer()
    rungs.append(("D5 Breast Cancer (30-D)", bc.data, bc.target))

    return rungs



def scale_for_geometry(X):
    scaler = StandardScaler()
    return scaler.fit_transform(X)


def pad_to_2d(Z):
    if Z.shape[1] >= 2:
        return Z[:, :2]
    zeros = np.zeros((Z.shape[0], 1))
    return np.hstack([Z, zeros])


def reconstruction_rmse(X, X_hat):
    return float(np.sqrt(np.mean((X - X_hat) ** 2)))


def plot_embedding_panels(results, metric_label):
    fig = plt.figure(figsize=(15, 6))
    for idx, item in enumerate(results):
        ax = fig.add_subplot(2, 5, idx + 1)
        Z2 = pad_to_2d(item["Z"])
        ax.scatter(Z2[:, 0], Z2[:, 1], c=item["y"], s=8, cmap="viridis", alpha=0.8)
        ax.set_title(item["name"].split("(")[0].strip(), fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    ax = fig.add_subplot(2, 1, 2)
    xs = np.arange(1, len(results) + 1)
    ys = [item["metric"] for item in results]
    ax.plot(xs, ys, marker="o")
    ax.set_xticks(xs)
    ax.set_xticklabels(["D" + str(i) for i in xs])
    ax.set_ylabel(metric_label)
    ax.set_title(metric_label + " vs ladder rung")
    fig.tight_layout()
    return fig


## The concept, built once: PCA whitening audit before ICA rotation

ICA starts with the lesson's linear algebra audit $$X_c=U\Sigma V^\top,\qquad Z=X_cV_r$$.
The toy matrix must still produce means $[2.50,2.50]$, singular values $[2.828,1.414]$, and PCA energy $0.800$ before the ICA-specific rotation begins.

In [ ]:

toy = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
    [4.0, 3.0],
])

centered = toy - toy.mean(axis=0)
U, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
energies = singular_values ** 2
first_share = energies[0] / energies.sum()

assert np.allclose(toy.mean(axis=0), [2.5, 2.5])
assert np.allclose(np.round(singular_values, 3), [2.828, 1.414])
assert np.isclose(np.round(first_share, 3), 0.800)

print("means", toy.mean(axis=0))
print("singular values", np.round(singular_values, 3))
print("first-component variance share", np.round(first_share, 3))


Now implement a real FastICA-style fixed-point rotation after whitening. The components may flip signs or swap order, so reconstruction and absolute correlations matter more than names.

In [ ]:

def symmetric_decorrelation(W):
    values, vectors = np.linalg.eigh(W @ W.T)
    return vectors @ np.diag(1.0 / np.sqrt(np.maximum(values, 1e-12))) @ vectors.T @ W


def ica_method(X, n_components=2, max_iter=200, tol=1e-5, random_state=7):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    Xc = X - mean
    U, singular_values, Vt = np.linalg.svd(Xc, full_matrices=False)
    whitening = Vt[:n_components].T / np.maximum(singular_values[:n_components], 1e-12) * np.sqrt(X.shape[0] - 1)
    X_white = Xc @ whitening
    rng = np.random.default_rng(random_state)
    W = rng.normal(size=(n_components, n_components))
    W = symmetric_decorrelation(W)
    for step in range(max_iter):
        WX = X_white @ W.T
        G = np.tanh(WX)
        G_prime = 1.0 - G ** 2
        W_new = (G.T @ X_white) / X.shape[0] - np.diag(G_prime.mean(axis=0)) @ W
        W_new = symmetric_decorrelation(W_new)
        distance = np.max(np.abs(np.abs(np.diag(W_new @ W.T)) - 1.0))
        W = W_new
        if distance < tol:
            break
    S = X_white @ W.T
    mixing = np.linalg.lstsq(S, Xc, rcond=None)[0]
    X_hat = S @ mixing + mean
    return S, X_hat, W, mixing

Z_ica, X_hat_ica, W_ica, mixing_ica = ica_method(toy, n_components=2, random_state=7)
assert Z_ica.shape == (4, 2)
assert np.isclose(np.round(first_share, 3), 0.800)
print("ICA toy source shape", Z_ica.shape)
print("ICA toy reconstruction rmse", np.round(reconstruction_rmse(toy, X_hat_ica), 6))


## The dataset ladder

Each rung is standardized before whitening so scale does not masquerade as independence.

In [ ]:

rungs = dimred_ladder()
for idx, (name, X, y) in enumerate(rungs, start=1):
    y_array = np.asarray(y)
    unique_preview = np.unique(y_array[: min(len(y_array), 200)])[:8]
    print(f"D{idx}: {name}")
    print(f"  X shape: {X.shape}")
    print(f"  y preview: {unique_preview}")
    print(f"  first row: {np.round(X[0, : min(6, X.shape[1])], 3)}")


## Run the same ICA method across D1-D5

In [ ]:

results = []
for name, X, y in dimred_ladder():
    X_scaled = scale_for_geometry(X)
    Z, X_hat, W, mixing = ica_method(X_scaled, n_components=2, random_state=7)
    rmse = reconstruction_rmse(X_scaled, X_hat)
    results.append({"name": name, "Z": Z, "y": y, "metric": rmse})

print("rung | reconstruction rmse")
for idx, item in enumerate(results, start=1):
    print(f"D{idx} | {item['metric']:.3f} | {item['name']}")


## Results visualization

Panels show the two recovered source coordinates; the curve reports reconstruction error across the ladder.

In [ ]:

fig = plot_embedding_panels(results, "reconstruction rmse")
plt.show()


## Pitfall on D5: sign and order instability

ICA components are not named ground truth. Two runs can flip signs or swap order while reconstructing the data similarly.

In [ ]:

name, X_d5, y_d5 = dimred_ladder()[-1]
X_d5_scaled = scale_for_geometry(X_d5)
Z_a, X_hat_a, W_a, mixing_a = ica_method(X_d5_scaled, n_components=2, random_state=7)
Z_b, X_hat_b, W_b, mixing_b = ica_method(X_d5_scaled, n_components=2, random_state=8)
correlation = np.corrcoef(Z_a.T, Z_b.T)[:2, 2:]
rmse_a = reconstruction_rmse(X_d5_scaled, X_hat_a)
rmse_b = reconstruction_rmse(X_d5_scaled, X_hat_b)

print("absolute source correlation across seeds")
print(np.round(np.abs(correlation), 3))
print("rmse seed 7", round(rmse_a, 3))
print("rmse seed 8", round(rmse_b, 3))


## Evaluate it + Practice

- Track the ladder metric: reconstruction RMSE, plus source correlation on D1 when true sources are known. Compare it with a no-skill baseline such as using the first two raw scaled columns or the input mean reconstruction.
- Sanity check shapes: D1-D5 should all return a two-column visualization embedding and a scalar metric.
- Ablation: compare component names instead of absolute correlation or reconstruction; the metric should get worse or the visualization should lose structure.
- Failure signals: tiny hyperparameter changes flip the pattern, D5 improves only on the training objective, or one raw-scale feature dominates.
- Stability check: rerun with a nearby seed or hyperparameter and compare reconstruction, trustworthiness, or neighbor preservation rather than component names.

Practice 1: Change the number of components or atoms and redraw the metric curve.

Practice 2: Create two synthetic source signals, mix them, and compare recovered sources.

Practice 3: Increase max_iter and see whether D5 stability improves.